In [ ]:
# Imports
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path: sys.path.insert(0, project_root)

from src import *
import scipy.sparse.linalg as spsl
import matplotlib.pyplot as plt
import matplotlib
from itertools import product

In [ ]:
# Hamiltonian Parameters
num_qubits = 12
J = 1
h = 1

# MODMD Parameters
max_energy_level = 3
kd_ratio = 2.5
noise_threshold = 1/(10**1.5)
# epsilon = 1e-2
epsilon = 1/(10**2.5)
K_values = np.arange(100,510,100)
num_modmd_observables = 7
num_trials = 20

# QMEGS Parameters
N_list = [int((K + K/kd_ratio)*np.log(num_modmd_observables)/(epsilon**2)) for K in K_values] 
K_dominant=4
gamma=0.5

In [ ]:
# Generate Hamiltonian and get true eigenenergies
unnormalized_hamiltonian = tfim_hamiltonian(num_qubits,J,h).to_matrix(sparse=True)
hamiltonian_norm = spsl.norm(unnormalized_hamiltonian,2)
sparse_hamiltonian = unnormalized_hamiltonian/hamiltonian_norm

v,w = np.linalg.eigh(sparse_hamiltonian.toarray())
true_eigenenergies = np.unique(np.round(v,8))[:max_energy_level+1]

In [ ]:
delta_t = .1 * hamiltonian_norm
T_list = [(K + K/kd_ratio)*delta_t for K in K_values]

# Construct reference state
indices = np.argsort(sparse_hamiltonian.diagonal())
reference_state = bitstring_superposition_state(num_qubits,[bin(indices[i])[2:] for i in [0,2,5,4,8,6]])

# Get evolved reference states
max_K = K_values[-1]
max_d = int(max_K/kd_ratio)
time_evolution_operator = -1j*sparse_hamiltonian*delta_t
evolved_reference_states = spsl.expm_multiply(time_evolution_operator,reference_state,start=0,stop=max_d+max_K+1,num = max_d+max_K+2)

modmd_eigenstate_overlaps = np.abs(w.conj().T @ reference_state)**2

In [ ]:
# MODMD Results
modmd_results = []

for trial in range(num_trials):
    
    modmd_observables = [SparsePauliOp('I' * num_qubits).to_matrix(sparse=True)] + random_one_local_paulis(num_qubits,num_modmd_observables-1)

    X_elements = generate_X_elements(modmd_observables,max_d,max_K,reference_state,evolved_reference_states)
    gaussian_noise = np.random.normal(0,epsilon,size=X_elements.shape) + 1j * np.random.normal(0,epsilon,size=X_elements.shape)
    noisy_X_elements = X_elements + gaussian_noise
    
    modmd_results.append(varying_K_results(len(modmd_observables),noise_threshold,noisy_X_elements,delta_t,K_values,kd_ratio,max_energy_level))

absolute_modmd_errors = np.array([np.abs(np.array(modmd_results[i]) - true_eigenenergies) for i in range(num_trials)])
modmd_average_errors = [np.average(absolute_modmd_errors,0)[:,energy_level] for energy_level in range(max_energy_level+1)]

In [ ]:
spectrum, population = v, modmd_eigenstate_overlaps

In [ ]:
# Run QMEGS
qmegs_results = {}
t_total_QMEGS = {}

alpha_values = [.5,1,5,10,50]
q_values = [.01,.05,.1,.5,1]

index_list = np.array([np.where(np.argsort(population)[::-1] == i)[0][0] for i in range(max_energy_level+1)])

for alpha, q in product(alpha_values,q_values):

    absolute_qmegs_errors = []
    t_total_QMEGS[(alpha,q)] = []

    for k in range(len(K_values)):
        
        T_max=T_list[k]
        d_x = q/T_max
        N = N_list[k]
        
        Z_est,t_list, _, T_total = generate_Z_fast(spectrum,population,T_max,N,gamma)
        t_total_QMEGS[(alpha,q)].append(np.sum(np.abs(t_list)))
        output_energy = QMEGS_new_fast(Z_est, d_x, t_list, max(index_list) + 1, alpha, T_max)
                
        errors = [min(np.abs(spectrum[l] - output_energy)) for l in range(K_dominant)]

        absolute_qmegs_errors.append(errors)

    qmegs_results[(alpha,q)] = np.array(absolute_qmegs_errors).T

In [ ]:
# Plot hyperparameter sweep
colors = get_color_set('TFIM')

fig, ax = plt.subplots(len(q_values),len(alpha_values), figsize=(15,10))

for i, alpha in enumerate(alpha_values):
    for j, dx_coeff in enumerate(q_values):
        for energy_level in range(max_energy_level+1):
            ax[j][i].semilogy(T_list, qmegs_results[(alpha,dx_coeff)][energy_level], color=colors[energy_level], 
                              label=r'$|\delta E_{%g}|$' % energy_level)
            ax[j][i].set_xticks([])
            ax[j][i].set_yticks([])
            
            if i == 0 and j == 0:
                ax[j][i].legend(loc='lower left', fontsize=10)
            if j == 0:
                ax[j][i].set_title(r"$\alpha_{\rm {QMEGS}}$=" + str(alpha))
            if i == 0:
                ax[j][i].set_ylabel(r"$q_{\rm {QMEGS}}$=" + str(dx_coeff), labelpad = 30)
            
fig.supylabel('Absolute Error', x = .1)
fig.supxlabel(r'Maximal Simulation Time/$\Delta t$',y=.05)

In [ ]:
# Select best alpha, dx_coeff based on total error across energy levels and time
total_errors_across_levels_and_time = [np.sum(qmegs_results[(alpha,q)]) for alpha, dx_coeff in product(alpha_values,q_values)]
best_hyperparameters = list(product(alpha_values,q_values))[np.argmin(total_errors_across_levels_and_time)]

In [ ]:
# Plot Error vs. t_max
alpha, q = best_hyperparameters

colors = get_color_set('TFIM')
matplotlib.rcParams.update({'font.size': 14})

for energy_level in range(max_energy_level+1):
    plt.semilogy(K_values + K_values/kd_ratio, qmegs_results[(alpha,q)][energy_level]/modmd_average_errors[energy_level], 
                 color=colors[energy_level], label=r'$E_{%g}$' % energy_level)

plt.xlabel(r'Maximal Simulation Time/$\Delta t$')
plt.ylabel('Enhancement')
plt.legend(loc='upper left', edgecolor='none', facecolor='none', fontsize=14)

In [ ]:
# Plot Error vs. t_total
alpha, q = best_hyperparameters

colors = get_color_set('TFIM')
matplotlib.rcParams.update({'font.size': 14})

for energy_level in range(max_energy_level+1):
    plt.semilogy((K_values + K_values/kd_ratio)*((K_values + K_values/kd_ratio)+1)*delta_t/(2*epsilon**2), modmd_average_errors[energy_level], color=colors[energy_level], label=r'$|\delta E_{%g}|$' % energy_level)
    plt.semilogy(t_total_QMEGS[(alpha,q)], qmegs_results[(alpha,dx_coeff)][energy_level], '--', color=colors[energy_level])

plt.xlabel(r'Total Simulation Time')
plt.ylabel('Absolute Error')
plt.legend(loc='upper right', edgecolor='none', facecolor='none', fontsize=14)
